# Preprocesamiento de datos

En este notebook se describe el proceso de preprocesamiento aplicado al conjunto de datos utilizado para la predicción de readmisión hospitalaria a 30 días. En este contexto, la readmisión se entiende como un nuevo ingreso hospitalario producido dentro de los 30 días posteriores al alta previa, siguiendo un criterio ampliamente utilizado en la literatura y en indicadores de calidad.

Las principales tareas que se realizarán en esta fase son:

- Limpieza de datos
- Tratamiento de valores faltantes
- Codificación de variables categóricas
- Transformación de variables
- Preparación del dataset final para el modelado

Para ello se creo el scrip `preprocessing.py`este se encarga de hacer una limpieza general. La idea es ir mirando si lo hace de manera correcta ajustando las funciones para que al final se puedan usar estas de manera definitiva para tener un mejor pipeline. El notebook se utiliza como apoyo para comprobar paso a paso que dichas funciones realizan correctamente el procesamiento y para justificar las decisiones adoptadas durante esta fase.

Debido al gran volumen del conjunto de datos MIMIC-IV, en esta primera versión del trabajo se seleccionaron únicamente las tablas más relevantes para el problema planteado, concretamente `patients`, `admissions`.

Esto se hizo ya que el ordenador desde donde se trabajo no presentaba suficiente RAM para procesar todo los datos y hacer merge de las tablas

A partir de estas fuentes se construyó un dataset base que combina la información demográfica del paciente con los datos asociados a cada hospitalización. Posteriormente, se generó la variable objetivo de readmisión hospitalaria en función de la secuencia temporal de ingresos de cada paciente.



El proceso de preprocesamiento se dividió en dos etapas principales.

En la **primera etapa** se llevó a cabo una limpieza estructural del dataset, incluyendo la eliminación de duplicados, la conversión de variables temporales y la generación de variables derivadas, como la duración de la estancia hospitalaria. Además, se integró la información procedente de las tablas de pacientes y admisiones. Esta parte se encapsuló en la función `run_preprocessing_part1`.

En la **segunda etapa** se abordó el tratamiento de valores faltantes, la eliminación de variables no adecuadas para el modelado, la codificación de variables categóricas y la preparación del dataset final para su uso en los modelos Esta fase quedó implementada en la función `run_preprocessing_part2`.



# 1. Limpieza estructural y construcción del dataset base

En esta primera parte se comprueba el funcionamiento de `run_preprocessing_part1`, responsable de generar un dataset limpio a partir de las tablas originales y de crear variables derivadas relevantes para el análisis posterior.# 1 PARTE

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.data.preprocessing import  run_preprocessing_part1

df = run_preprocessing_part1()

df.head()

Loading datasets...
Loading patients...
Loading admissions...
Loading diagnoses...
Cleaning datasets...
Merging datasets...
Saving interim dataset...
Preprocessing completed.


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,race,edregtime,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN


In [5]:
df.shape


(546028, 22)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int64         
 1   hadm_id               546028 non-null  int64         
 2   admittime             546028 non-null  datetime64[ns]
 3   dischtime             546028 non-null  datetime64[ns]
 4   deathtime             11790 non-null   object        
 5   admission_type        546028 non-null  object        
 6   admit_provider_id     546024 non-null  object        
 7   admission_location    546027 non-null  object        
 8   discharge_location    396210 non-null  object        
 9   insurance             536673 non-null  object        
 10  language              545253 non-null  object        
 11  marital_status        532409 non-null  object        
 12  race                  546028 non-null  object        
 13 

In [ ]:
df.head()

<bound method NDFrame.head of         subject_id   hadm_id           admittime           dischtime  \
0         10000032  22595853 2180-05-06 22:23:00 2180-05-07 17:15:00   
1         10000032  22841357 2180-06-26 18:27:00 2180-06-27 18:49:00   
2         10000032  25742920 2180-08-05 23:44:00 2180-08-07 17:50:00   
3         10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00   
4         10000068  25022803 2160-03-03 23:16:00 2160-03-04 06:26:00   
...            ...       ...                 ...                 ...   
546023    19999828  25744818 2149-01-08 16:44:00 2149-01-18 17:00:00   
546024    19999828  29734428 2147-07-18 16:23:00 2147-08-04 18:10:00   
546025    19999840  21033226 2164-09-10 13:47:00 2164-09-17 13:42:00   
546026    19999840  26071774 2164-07-25 00:27:00 2164-07-28 12:15:00   
546027    19999987  23865745 2145-11-02 21:38:00 2145-11-11 12:57:00   

                  deathtime  admission_type admit_provider_id  \
0                       NaN          URG

In [24]:
df.isnull().sum().sort_values(ascending=False)

deathtime               534238
dod                     401062
edregtime               166788
edouttime               166788
discharge_location      149818
marital_status           13619
insurance                 9355
language                   775
admit_provider_id            4
admission_location           1
subject_id                   0
dischtime                    0
admittime                    0
hadm_id                      0
admission_type               0
race                         0
length_of_stay               0
hospital_expire_flag         0
gender                       0
anchor_age                   0
anchor_year                  0
anchor_year_group            0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df["length_of_stay"].describe()

count    546028.000000
mean          4.221033
std           7.201291
min          -1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         515.000000
Name: length_of_stay, dtype: float64

In [8]:
df["subject_id"].nunique()

223452

In [25]:
import os
os.listdir("../data/interim")

['.gitkeep', 'cleaned_dataset.csv']

In [7]:
summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "n_patients": df["subject_id"].nunique(),
    "n_admissions": df["hadm_id"].nunique()
}

summary

{'n_rows': 546028,
 'n_columns': 22,
 'n_patients': 223452,
 'n_admissions': 546028}

# 2. Preparación del conjunto final para modelado

En esta segunda parte se parte del dataset limpio `cleaned_dataset` para aplicar las transformaciones finales orientadas al modelado. Entre ellas se incluyen la eliminación de variables no relevantes, el tratamiento de valores faltantes y la codificación de variables categóricas.

Trabajar desde este dataset intermedio permite reutilizar una versión ya depurada de los datos sin repetir todo el procesamiento inicial, lo que mejora el pipeline.

# Creación varaible 30 dias

bla bla

# Eliminación de variables no adecuadas e imputación de valores categóricos

Antes de entrenar los modelos es necesario revisar qué variables deben mantenerse y cuáles conviene eliminar. En esta fase se excluyen columnas que:

- no aportan información útil para la predicción,
- presentan riesgo de fuga de información,
- resultan redundantes con otras variables ya derivadas,
- o describen eventos demasiado específicos para el objetivo del modelo.

Además, se imputan los valores faltantes en variables categóricas con una categoría explícita como `Unknown`, de forma que la ausencia de información no obligue a eliminar registros potencialmente útiles.

Algunas variables, como `deathtime`, `dod`, `edregtime` o `edouttime`, se eliminan por no resultar adecuadas para esta versión del problema predictivo. 

También se eliminan variables temporales originales como `admittime` y `dischtime` una vez que ya se ha derivado la característica `length_of_stay`, que resume de forma más directa la duración del ingreso.

En el caso de variables categóricas con un número moderado de valores faltantes, como `insurance`, `marital_status` o `language`, se utiliza la categoría `Unknown` para conservar la información de cara ak modelo sin introducir imputacioness.

# Codificación de variables categóricas

Una vez tratadas las categorías faltantes, las variables de tipo objeto se transforman a formato numérico mediante codificación *one-hot*. Este enfoque permite representar variables categóricas con múltiples valores sin imponer un orden artificial entre categorías.

# Validación del pipeline implementado en `run_preprocessing_part2`


In [35]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
from src.config import DATA_INTERIM
from src.data.preprocessing import run_preprocessing_part2


df =pd.read_csv(DATA_INTERIM/"cleaned_dataset.csv",
     parse_dates=["admittime","dischtime"]
     )
df.columns

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'length_of_stay',
       'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype='object')

In [ ]:
df_model = run_preprocessing_part2(df.copy())
print("Columnas:", len(df_model.columns))

Preparing dataset for modeling...
Creating readmission target variable...
Columns before processing: 24
Columns after encoding: 97
Saving processed dataset...
Model dataset created.
Columnas función: 97


In [38]:
df_model[["previous_admissions","readmission_30_days"]].head(10)

,previous_admissions,readmission_30_days
0,0,0
1,1,1
3,2,1
5,0,0
8,0,0
15,0,0
19,0,0
16,1,0
17,2,0
22,0,1


In [39]:
df_model["readmission_30_days"].value_counts()
df_model.head()

,discharge_location,length_of_stay,anchor_age,previous_admissions,readmission_30_days,gender_F,gender_M,race_AMERICAN INDIAN/ALASKA NATIVE,race_ASIAN,race_ASIAN - ASIAN INDIAN,...,admission_location_CLINIC REFERRAL,admission_location_EMERGENCY ROOM,admission_location_INFORMATION NOT AVAILABLE,admission_location_INTERNAL TRANSFER TO OR FROM PSYCH,admission_location_PACU,admission_location_PHYSICIAN REFERRAL,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL
0,HOME,0,52,0,0,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,HOME,1,52,1,1,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,HOME,2,52,2,1,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
5,HOME HEALTH CARE,4,72,0,0,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
8,Unknown,0,48,0,0,True,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


In [40]:
len(df_model.columns)


97